# 🔍 El caso de los Retrievers · Live coding
**Orquestación de flujos con LangChain y LlamaIndex — Retrievers**
Autora: Elena Díaz - [@HelenDiMo](https://github.com/HelenDiMo)

Lord Edmund Aldridge ha aparecido muerto en su biblioteca de Marlowe Hall. Tenemos un **expediente de 11 documentos** (testimonios, un recibo, una carta…). Holmes necesita un ayudante que sepa **salir a buscar las pruebas correctas**: ese ayudante es el **retriever**.

> 🧠 **La idea en una frase:** un retriever recibe una pregunta y devuelve los documentos más útiles para responderla. El LLM (Holmes) solo razona con lo que el retriever le trae. Si trae las pruebas equivocadas, el mejor detective del mundo se equivoca.

| Parte | Qué veremos | ⏱ |
|---|---|---|
| 0 | Preparación y el expediente | 2 min |
| 1 | Retriever por **similitud** (significado) | 5 min |
| 2 | **BM25** (palabras exactas) | 4 min |
| 3 | **MMR** (pruebas variadas) | 3 min |
| 4 | **Híbrido**: lo mejor de los dos | 4 min |
| 5 | La **cadena completa** (LCEL) | 2 min |

## Parte 0 · Preparación

In [1]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Permite importar el paquete `sherlock` (carpeta src/) desde el notebook
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from importlib.metadata import version
from sherlock.embeddings import describir_backend
from sherlock.llm import describir_detective

for paquete in ["langchain-core", "langchain-community", "langchain-classic",
                "llama-index-core", "llama-index-retrievers-bm25"]:
    print(f"{paquete:<30} {version(paquete)}")
print("Embeddings:", describir_backend())
print("Detective :", describir_detective())

langchain-core                 1.6.9
langchain-community            0.4.2
langchain-classic              1.0.8
llama-index-core               0.14.25
llama-index-retrievers-bm25    0.8.0
Embeddings: spaCy es_core_news_md (vectores de palabras promediados)
Detective : detective simulado (sin LLM)


### El expediente
Cada documento es corto, así que **cada uno es un único "trozo"** (*chunk*). En un proyecto real, los documentos largos se trocean antes de indexarlos (por ejemplo con `RecursiveCharacterTextSplitter`); aquí nos lo ahorramos.

In [2]:
from sherlock.expediente import cargar_expediente

expediente = cargar_expediente()

for d in expediente:
    print(f"{d['id']:<11} {d['tipo']:<10} {d['titulo']}")

print()
print(expediente[0]["texto"])

forense     informe    Informe del forense
mayordomo   testimonio Testimonio del mayordomo
jardinero   testimonio Testimonio del jardinero
cocinera    testimonio Testimonio de la cocinera
doncella    testimonio Testimonio de la doncella
botica      recibo     Recibo de la botica
carta       carta      Carta de Lord Aldridge a su abogado
telegrama   telegrama  Telegrama del inspector Lestrade
watson      diario     Diario del doctor Watson
club        registro   Registro de salida del club Albemarle
cochero     testimonio Testimonio del cochero

Lord Edmund Aldridge fue hallado sin vida en la biblioteca de Marlowe Hall. La muerte se produjo entre las diez y las once de la noche. No hay heridas ni señales de violencia. En la copa de brandy encontré restos de extracto de belladona, una dosis suficiente para detener el corazón en menos de una hora. La ventana de la biblioteca estaba cerrada por dentro y la puerta no había sido forzada.


## Parte 1 · Retriever por similitud (embeddings)

**¿Qué es?** Cada texto se convierte en una lista de números (un *embedding*) que representa su **significado**. Dos textos que hablan de lo mismo tienen números parecidos, aunque usen palabras distintas.

**¿Por qué importa?** Permite encontrar "perro" cuando buscas "sabueso", o "veneno" cuando buscas "sustancia tóxica".

**¿Cómo se usa?** Se guarda todo en un *vector store* y se le pide un retriever con `as_retriever`. El parámetro clave es **`k`**: cuántos documentos quieres que traiga.

### 1.1 En LangChain

In [3]:
from langchain_core.vectorstores import InMemoryVectorStore
from sherlock.expediente import a_documentos_langchain
from sherlock.embeddings import crear_embeddings_langchain
from sherlock.vista import mostrar

docs_lc = a_documentos_langchain(expediente)
vectorstore = InMemoryVectorStore.from_documents(docs_lc, crear_embeddings_langchain())

retriever_sim = vectorstore.as_retriever(search_kwargs={"k": 3})

mostrar(retriever_sim.invoke("el sabueso"), "LangChain · 'el sabueso'")

LangChain · 'el sabueso'
  1. [jardinero ] puntuación       - | Sombra, el lebrel de la casa, no ladró aquella noche cuando alguien cruzó el …
  2. [watson    ] puntuación       - | Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí,…
  3. [doncella  ] puntuación       - | Pasé por el pasillo a las diez y veinte y vi luz bajo la puerta de la bibliot…


Fíjate: **la palabra "sabueso" no aparece en ningún documento** (el jardinero habla de un *lebrel* y de un *perro*), y aun así el primer resultado es el testimonio del jardinero. Eso es buscar por significado.

El retriever de LangChain no enseña las puntuaciones. Si las quieres, las pides al vector store:

In [4]:
for doc, puntuacion in vectorstore.similarity_search_with_score("el sabueso", k=3):
    print(f"{puntuacion:.3f}  {doc.metadata['id']}")

0.464  jardinero
0.294  watson
0.225  doncella


### 1.2 En LlamaIndex

In [5]:
from llama_index.core import VectorStoreIndex, Settings
from sherlock.expediente import a_documentos_llamaindex
from sherlock.embeddings import crear_embeddings_llamaindex

Settings.embed_model = crear_embeddings_llamaindex()
Settings.llm = None   # aquí solo buscamos; no generamos respuestas

indice = VectorStoreIndex.from_documents(a_documentos_llamaindex(expediente))

retriever_sim_li = indice.as_retriever(similarity_top_k=3)

mostrar(retriever_sim_li.retrieve("el sabueso"), "LlamaIndex · 'el sabueso'")

LLM is explicitly disabled. Using MockLLM.
LlamaIndex · 'el sabueso'
  1. [jardinero ] puntuación   0.464 | Sombra, el lebrel de la casa, no ladró aquella noche cuando alguien cruzó el …
  2. [watson    ] puntuación   0.294 | Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí,…
  3. [doncella  ] puntuación   0.225 | Pasé por el pasillo a las diez y veinte y vi luz bajo la puerta de la bibliot…


Mismo resultado, distinto estilo:

| | LangChain | LlamaIndex |
|---|---|---|
| Crear | `as_retriever(search_kwargs={"k": 3})` | `as_retriever(similarity_top_k=3)` |
| Preguntar | `.invoke(pregunta)` | `.retrieve(pregunta)` |
| Devuelve | `Document` | `NodeWithScore` (con la puntuación incluida) |

### 1.3 Experimenta con `k`

In [6]:
for k in (1, 3, 6):
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    print(f"k={k}:", [d.metadata["id"] for d in retriever.invoke("sustancia tóxica")])

k=1: ['botica']
k=3: ['botica', 'forense', 'watson']
k=6: ['botica', 'forense', 'watson', 'jardinero', 'carta', 'telegrama']


> 🤔 **Mini-pregunta:** si `k` es muy pequeño, ¿qué riesgo hay? ¿Y si es muy grande?
>
> <details><summary>Ver respuesta</summary>
>
> Con `k` pequeño te puedes dejar fuera una prueba clave. Con `k` grande le das a Holmes ruido y le haces leer (y pagar) de más, y las pruebas buenas se diluyen entre las malas.
> </details>

## Parte 2 · BM25: buscar por palabras exactas

**¿Qué es?** El buscador "de toda la vida": puntúa los documentos según cuántas palabras de tu pregunta contienen, dando más valor a las palabras **raras**.

**¿Por qué importa?** Los embeddings fallan con nombres propios y términos poco comunes que no han visto. BM25 los clava.

Probemos las dos preguntas "trampa" con **ambos** métodos:

In [7]:
from langchain_community.retrievers import BM25Retriever
from sherlock.texto import tokenizar

bm25_de_serie = BM25Retriever.from_documents(docs_lc, k=3, preprocess_func=tokenizar)

for pregunta in ["Whitcombe", "el sabueso"]:
    print("=" * 70)
    print("PREGUNTA:", pregunta)
    mostrar(retriever_sim.invoke(pregunta), "  Similitud:")
    mostrar(bm25_de_serie.invoke(pregunta), "  BM25 de serie:")

PREGUNTA: Whitcombe
  Similitud:
  1. [telegrama ] puntuación       - | Holmes, el señor Hale asegura que pasó toda la noche en el club Albemarle. Ti…
  2. [carta     ] puntuación       - | Estimado señor Whitcombe: he descubierto que mi secretario ha falsificado las…
  3. [doncella  ] puntuación       - | Pasé por el pasillo a las diez y veinte y vi luz bajo la puerta de la bibliot…
  BM25 de serie:
  1. [carta     ] puntuación       - | Estimado señor Whitcombe: he descubierto que mi secretario ha falsificado las…
  2. [club      ] puntuación       - | Libro de salidas, noche del catorce de noviembre. El señor R. Hale firmó su s…
  3. [cochero   ] puntuación       - | Esperaba en el patio con el coche a las diez y media y vi luz en la ventana d…
PREGUNTA: el sabueso
  Similitud:
  1. [jardinero ] puntuación       - | Sombra, el lebrel de la casa, no ladró aquella noche cuando alguien cruzó el …
  2. [watson    ] puntuación       - | Holmes examinó la licorera con su lupa y concluyó q

Dos fallos, uno para cada método:

* **"Whitcombe"** (el abogado, un apellido raro): BM25 encuentra la carta. La similitud no conoce esa palabra y devuelve lo que le sale. Compruébalo con las puntuaciones y con la lista de palabras desconocidas:

In [8]:
from sherlock.embeddings import palabras_desconocidas

print("Palabras que el modelo no conoce:", palabras_desconocidas("Whitcombe y el sabueso"))
print()
print("Puntuaciones de similitud para 'Whitcombe' (comparar con 0.46 de 'el sabueso'):")
for doc, puntuacion in vectorstore.similarity_search_with_score("Whitcombe", k=3):
    print(f"  {puntuacion:.3f}  {doc.metadata['id']}")

Palabras que el modelo no conoce: ['Whitcombe']

Puntuaciones de similitud para 'Whitcombe' (comparar con 0.46 de 'el sabueso'):
  0.116  telegrama
  0.077  carta
  0.012  doncella


Las puntuaciones son bajísimas: es **ruido**. Si la carta aparece, es por casualidad.

* **"el sabueso"**: BM25 no encuentra nada… **pero devuelve 3 documentos igualmente**, con puntuación 0. ¡Son de relleno! El BM25 de serie siempre devuelve `k` resultados, aunque ninguno tenga que ver. Cuando lo mezclemos con otro retriever, ese relleno nos estropearía el resultado.

La solución es un BM25 con **corte**: si no hay coincidencias, no devuelve nada. Lo tenemos ya preparado en `sherlock.retrievers`:

In [9]:
from sherlock.retrievers import crear_bm25_langchain

retriever_bm25 = crear_bm25_langchain(docs_lc, k=3)

mostrar(retriever_bm25.invoke("Whitcombe"), "BM25 con corte · 'Whitcombe'")
mostrar(retriever_bm25.invoke("el sabueso"), "BM25 con corte · 'el sabueso'")

BM25 con corte · 'Whitcombe'
  1. [carta     ] puntuación   2.051 | Estimado señor Whitcombe: he descubierto que mi secretario ha falsificado las…
BM25 con corte · 'el sabueso'
   (no ha devuelto ningún documento)


### ¿Y si quiero hacer mi propio retriever?
Un retriever es solo **"entra una pregunta, salen documentos"**. Este es el más pequeño posible (devuelve los documentos que contienen el texto tal cual):

In [10]:
from langchain_core.retrievers import BaseRetriever

class RetrieverMinimo(BaseRetriever):
    docs: list

    def _get_relevant_documents(self, query, *, run_manager):
        return [d for d in self.docs if query.lower() in d.page_content.lower()]

mostrar(RetrieverMinimo(docs=docs_lc).invoke("belladona"), "Retriever propio · 'belladona'")

Retriever propio · 'belladona'
  1. [forense   ] puntuación       - | Lord Edmund Aldridge fue hallado sin vida en la biblioteca de Marlowe Hall. L…
  2. [botica    ] puntuación       - | Botica Pemberton e Hijos. Se despachan diez gramos de extracto de belladona, …


## Parte 3 · MMR: pruebas variadas

**¿Qué es?** *Maximal Marginal Relevance*. Es una búsqueda por similitud que, al elegir cada nuevo resultado, **penaliza los que se parecen a los ya elegidos**.

**¿Por qué importa?** Mira lo que pasa con una pregunta que tiene tres testimonios casi iguales:

In [11]:
pregunta = "¿Quién vio luz en la biblioteca?"

retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": 0.5},
)

mostrar(retriever_sim.invoke(pregunta), "Similitud (k=3):")
print()
mostrar(retriever_mmr.invoke(pregunta), "MMR (k=3):")

Similitud (k=3):
  1. [cochero   ] puntuación       - | Esperaba en el patio con el coche a las diez y media y vi luz en la ventana d…
  2. [cocinera  ] puntuación       - | Desde la cocina vi luz en la biblioteca a las diez y cuarto. La luz seguía en…
  3. [doncella  ] puntuación       - | Pasé por el pasillo a las diez y veinte y vi luz bajo la puerta de la bibliot…

MMR (k=3):
  1. [cochero   ] puntuación       - | Esperaba en el patio con el coche a las diez y media y vi luz en la ventana d…
  2. [forense   ] puntuación       - | Lord Edmund Aldridge fue hallado sin vida en la biblioteca de Marlowe Hall. L…
  3. [watson    ] puntuación       - | Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí,…


Con similitud, los 3 resultados son **testimonios casi idénticos** (luz encendida hasta tarde): gastas tu `k` en repetir lo mismo. Con MMR entra información **nueva** (el forense, el diario de Watson).

Dos mandos para jugar:
* **`lambda_mult`**: `1.0` = solo relevancia (igual que similitud), `0.0` = solo variedad.
* **`fetch_k`**: cuántos candidatos mira MMR antes de elegir los `k` finales.

In [12]:
for lambda_mult in (1.0, 0.7, 0.5, 0.3):
    r = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": lambda_mult},
    )
    print(f"lambda_mult={lambda_mult}:", [d.metadata["id"] for d in r.invoke(pregunta)])

lambda_mult=1.0: ['cochero', 'cocinera', 'doncella']
lambda_mult=0.7: ['cochero', 'cocinera', 'doncella']
lambda_mult=0.5: ['cochero', 'forense', 'watson']
lambda_mult=0.3: ['cochero', 'forense', 'watson']


**MMR en LlamaIndex.** La idea es la misma y el mando se llama `mmr_threshold` (`1.0` = solo relevancia, como `lambda_mult=1.0`). Pero **no es idéntico a LangChain**: LlamaIndex solo compara cada candidato con el *último* documento elegido, no con todos los anteriores. Con un expediente tan pequeño eso se nota, así que no esperes los mismos resultados. Además, la puntuación que ves ya está *ajustada por MMR* (no es la similitud pura).

In [13]:
retriever_mmr_li = indice.as_retriever(
    similarity_top_k=3,
    vector_store_query_mode="mmr",
    vector_store_kwargs={"mmr_threshold": 0.6},
)
mostrar(retriever_mmr_li.retrieve(pregunta), "LlamaIndex · MMR:")

LlamaIndex · MMR:
  1. [cochero   ] puntuación   0.390 | Esperaba en el patio con el coche a las diez y media y vi luz en la ventana d…
  2. [forense   ] puntuación   0.053 | Lord Edmund Aldridge fue hallado sin vida en la biblioteca de Marlowe Hall. L…
  3. [cocinera  ] puntuación   0.194 | Desde la cocina vi luz en la biblioteca a las diez y cuarto. La luz seguía en…


## Parte 4 · Híbrido: lo mejor de los dos mundos

**¿Qué es?** Un retriever que **combina varios retrievers**. Hasta ahora hemos visto que cada método falla donde el otro acierta, así que… ¡juntémoslos!

**¿Cómo mezcla los resultados?** No suma las puntuaciones (las de BM25 y las de similitud no son comparables). Usa **RRF** (*Reciprocal Rank Fusion*): mira solo la **posición** de cada documento en cada lista.

> Cada documento recibe `peso / (posición + 60)` por cada lista en la que aparece, y se suma todo.

Hagámoslo a mano para entenderlo, con la pregunta que mezcla una palabra rara y un concepto: **"Whitcombe y el sabueso"**.

In [14]:
def rrf(listas, pesos, c=60):
    puntos = {}
    for lista, peso in zip(listas, pesos):
        for posicion, id_doc in enumerate(lista, start=1):
            puntos[id_doc] = puntos.get(id_doc, 0) + peso / (posicion + c)
    return sorted(puntos.items(), key=lambda par: -par[1])

pregunta = "Whitcombe y el sabueso"
de_bm25 = [d.metadata["id"] for d in retriever_bm25.invoke(pregunta)]
de_similitud = [d.metadata["id"] for d in retriever_sim.invoke(pregunta)]

print("BM25     :", de_bm25)
print("Similitud:", de_similitud)
print()
for id_doc, puntos in rrf([de_bm25, de_similitud], [0.5, 0.5]):
    print(f"{puntos:.4f}  {id_doc}")

BM25     : ['carta']
Similitud: ['jardinero', 'watson', 'doncella']

0.0082  carta
0.0082  jardinero
0.0081  watson
0.0079  doncella


`carta` (la nº 1 de BM25) y `jardinero` (la nº 1 de similitud) quedan arriba, **empatadas** porque cada una es la primera de su lista: justo las dos pruebas que necesitamos. Ahora lo mismo con el retriever que ya trae LangChain:

> ⚠️ **Cuidado con los imports:** en LangChain 1.x, `EnsembleRetriever` **ya no está en `langchain.retrievers`**. Ahora vive en `langchain_classic.retrievers`. Si ves `ModuleNotFoundError`, es esto.

In [15]:
from langchain_classic.retrievers import EnsembleRetriever

retriever_hibrido = EnsembleRetriever(
    retrievers=[retriever_bm25, retriever_sim],
    weights=[0.5, 0.5],     # cuánto cuenta cada uno
)

resultado = retriever_hibrido.invoke(pregunta)
print(f"El híbrido devuelve {len(resultado)} documentos (no 3)")
mostrar(resultado[:3], "Los 3 primeros:", puntuaciones=False)

El híbrido devuelve 4 documentos (no 3)
Los 3 primeros:
  1. [carta     ] puntuación       - | Estimado señor Whitcombe: he descubierto que mi secretario ha falsificado las…
  2. [jardinero ] puntuación       - | Sombra, el lebrel de la casa, no ladró aquella noche cuando alguien cruzó el …
  3. [watson    ] puntuación       - | Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí,…


Otro detalle: el `EnsembleRetriever` **no recorta a `k`**; devuelve la unión de las listas de cada retriever. Por eso hay que quedarse con `[:3]`.

### El híbrido en LlamaIndex: `QueryFusionRetriever`

In [16]:
from llama_index.core.retrievers import QueryFusionRetriever
from sherlock.retrievers import crear_bm25_llamaindex

nodos = list(indice.docstore.docs.values())
retriever_bm25_li = crear_bm25_llamaindex(nodos, k=3)

retriever_hibrido_li = QueryFusionRetriever(
    [retriever_sim_li, retriever_bm25_li],
    similarity_top_k=3,
    num_queries=1,            # 1 = no reescribe la pregunta, así no hace falta un LLM
    mode="reciprocal_rerank", # el mismo RRF
    use_async=False,
)

mostrar(retriever_hibrido_li.retrieve(pregunta), "LlamaIndex · híbrido:")

LlamaIndex · híbrido:
  1. [jardinero ] puntuación   0.017 | Sombra, el lebrel de la casa, no ladró aquella noche cuando alguien cruzó el …
  2. [carta     ] puntuación   0.017 | Estimado señor Whitcombe: he descubierto que mi secretario ha falsificado las…
  3. [watson    ] puntuación   0.016 | Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí,…


Las puntuaciones salen pequeñitas (~0.03) porque son puntos de RRF, **no** similitudes: no se pueden comparar con las de antes.

> 💡 Si pones `num_queries` mayor que 1, un LLM reescribe tu pregunta de varias formas y fusiona los resultados (*multi-query*). Es una técnica avanzada que queda fuera de esta píldora.

Un último truco: los **pesos**. Cambia cuánto cuenta cada retriever. Con la pregunta "¿Dónde pasó la noche Hale?", mira qué pasa cuando damos mucho peso a las palabras exactas (BM25 pone el telegrama el primero, porque contiene "noche" y "Hale") o a la similitud:

In [17]:
for pesos in ([0.5, 0.5], [0.8, 0.2], [0.2, 0.8]):
    r = EnsembleRetriever(retrievers=[retriever_bm25, retriever_sim], weights=pesos)
    print(f"pesos BM25/similitud = {pesos}:", [d.metadata["id"] for d in r.invoke("¿Dónde pasó la noche Hale?")[:3]])

pesos BM25/similitud = [0.5, 0.5]: ['club', 'mayordomo', 'telegrama']
pesos BM25/similitud = [0.8, 0.2]: ['club', 'mayordomo', 'telegrama']
pesos BM25/similitud = [0.2, 0.8]: ['club', 'mayordomo', 'doncella']


## Parte 5 · La cadena completa (orquestación con LCEL)

Todos los retrievers de LangChain son **Runnables**: piezas que se encadenan con `|`. Así se monta el flujo completo:

`pregunta → retriever → formatear las pruebas → prompt → LLM → texto`

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from sherlock.llm import crear_detective_langchain

def formatear(docs):
    return "\n".join(f"- [{d.metadata['titulo']}] {d.page_content}" for d in docs[:3])

prompt = ChatPromptTemplate.from_template("""Eres Sherlock Holmes. Responde SOLO con las pruebas que se te dan.
Si no bastan para responder, dilo.

PRUEBAS:
{contexto}

PREGUNTA: {pregunta}""")

cadena = (
    {"contexto": retriever_hibrido | formatear, "pregunta": RunnablePassthrough()}
    | prompt
    | crear_detective_langchain()
    | StrOutputParser()
)

print(cadena.invoke("¿Qué prueba demuestra que quien envenenó conocía la casa?"))

[Detective simulado: no es un LLM, solo muestra lo que recibió]
Me han llegado 3 pruebas:
- [Testimonio del jardinero] Sombra, el lebrel de la casa, no ladró aquella noche cuando alguien cruzó el jardín hacia la puerta de servicio. Y ese perro ladra a todo el que no conoce, incluso al cartero. Yo estaba en la caseta de las herramientas y lo oí todo: ni un solo ladrido. Quien fuese, el animal ya lo conocía de sobra.
- [Diario del doctor Watson] Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí, y no en la copa, antes de las nueve y media. Lo hizo alguien que conocía bien la rutina de la casa. Junto a la puerta de servicio había barro con arena rojiza, la misma que cubre el camino de la finca.
- [Testimonio de la doncella] Pasé por el pasillo a las diez y veinte y vi luz bajo la puerta de la biblioteca. Pensé que Su señoría seguía leyendo y no llamé para no molestarle. No escuché voces ni ruidos dentro. Todo estaba en silencio y la casa parecía tranquila, igual

Mira "qué le llegó" a Holmes antes de culpar al LLM de una mala respuesta. **Es lo primero que hay que revisar**:

In [19]:
print((retriever_hibrido | formatear).invoke("¿Qué prueba demuestra que quien envenenó conocía la casa?"))

- [Testimonio del jardinero] Sombra, el lebrel de la casa, no ladró aquella noche cuando alguien cruzó el jardín hacia la puerta de servicio. Y ese perro ladra a todo el que no conoce, incluso al cartero. Yo estaba en la caseta de las herramientas y lo oí todo: ni un solo ladrido. Quien fuese, el animal ya lo conocía de sobra.
- [Diario del doctor Watson] Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí, y no en la copa, antes de las nueve y media. Lo hizo alguien que conocía bien la rutina de la casa. Junto a la puerta de servicio había barro con arena rojiza, la misma que cubre el camino de la finca.
- [Testimonio de la doncella] Pasé por el pasillo a las diez y veinte y vi luz bajo la puerta de la biblioteca. Pensé que Su señoría seguía leyendo y no llamé para no molestarle. No escuché voces ni ruidos dentro. Todo estaba en silencio y la casa parecía tranquila, igual que cualquier otra noche.


> ℹ️ Si no tienes clave de API, el "detective" es una simulación que solo enumera las pruebas (no razona). Para usar un LLM de verdad, define la variable de entorno `ANTHROPIC_API_KEY` e instala `langchain-anthropic`. Lo importante de esta píldora es **qué pruebas le llegan**.

## 🧾 Chuleta final

| Necesito… | Uso | LangChain | LlamaIndex |
|---|---|---|---|
| Buscar por **significado** | Similitud | `vectorstore.as_retriever(search_kwargs={"k": 3})` | `index.as_retriever(similarity_top_k=3)` |
| Buscar **palabras exactas** (nombres, códigos) | BM25 | `BM25Retriever` (¡ojo al relleno!) | `BM25Retriever` (paquete aparte) |
| Resultados **variados** | MMR | `search_type="mmr"` | `vector_store_query_mode="mmr"` |
| **Lo mejor de ambos** | Híbrido (RRF) | `EnsembleRetriever` (en `langchain_classic`) | `QueryFusionRetriever` |

## ➡️ Ahora te toca a ti
Abre **`02_reto_TODO.ipynb`**: cinco pistas, cinco retrievers por elegir. Cuando las resuelvas todas, se revela el caso. 🕵️